# Inferencia: ¿real o generada por IA?

Este notebook carga el modelo exportado por `comparacion_efficientnet.ipynb`
(el archivo `best_model_real_vs_ia_v4.pt`, un ViT-B/16) y lo usa para clasificar imágenes nuevas.

**Requisitos previos:**
1. Tener el archivo `best_model_real_vs_ia_v4.pt` disponible (subido a este entorno, en Drive o con un enlace de descarga).
2. No hace falta GPU: la inferencia sobre pocas imágenes es rápida en CPU.

El archivo trae todo lo necesario para predecir igual que en la evaluación del trabajo:
arquitectura, pesos, preprocesamiento (lado corto al tamaño de entrada y recorte central),
normalización, umbral elegido en validación y mapeo de etiquetas.


In [ ]:
# Dependencias mínimas (mucho más liviano que el notebook de entrenamiento)
!pip install -q timm pillow torch torchvision gdown


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import torch
import torch.nn.functional as F
import timm
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando device: {DEVICE}")


## 1. Cargar el modelo guardado

Puedes usar una ruta local (si subiste el archivo directamente a este entorno) o pegar el
link para compartir de Google Drive -- el notebook lo descarga automáticamente.

**Si usas un link de Drive**, asegúrate de que el archivo tenga permisos
**"Cualquiera con el link puede ver"** (clic derecho sobre el archivo en Drive → Compartir →
Acceso general → "Cualquiera con el enlace"). Si el archivo es privado, la descarga va a fallar
aunque el link se vea bien.

In [ ]:
import re
import gdown

RAW_MODEL_PATH = "/content/drive/MyDrive/Proyecto grado Eafit/best_model_real_vs_ia_v4.pt"


def resolve_model_path(path: str) -> str:
    is_drive_link = path.startswith("http") and "drive.google.com" in path
    if not is_drive_link:
        return path

    match = re.search(r"/d/([a-zA-Z0-9_-]+)", path)
    if not match:
        raise ValueError(
            "No se pudo extraer el ID del archivo desde el link de Drive. "
            "Verifica que sea del formato https://drive.google.com/file/d/<ID>/view..."
        )
    file_id = match.group(1)
    local_path = "best_model_real_vs_ia_v4.pt"
    print(f"Detectado link de Google Drive -- descargando (ID: {file_id})...")
    gdown.download(id=file_id, output=local_path, quiet=False)
    return local_path


MODEL_PATH = resolve_model_path(RAW_MODEL_PATH)

assert os.path.exists(MODEL_PATH), (
    f"No se encontró '{MODEL_PATH}'. Sube el archivo a este entorno "
    "(panel de archivos a la izquierda en Colab), ajusta la ruta, o revisa "
    "que el link de Drive sea correcto y tenga permisos de acceso ('Cualquiera con el link')."
)

artifact = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

print("Arquitectura:", artifact["model_name"])
print("Tamaño de imagen esperado:", artifact["image_size"])
print("Mapeo de etiquetas:", artifact["label_mapping"])
print("Preprocesamiento:", artifact.get("preprocessing"))
print("Umbral elegido en validación:", artifact["threshold"])
print("Métricas en la prueba con ese umbral:")
for k, v in artifact["test_metrics_threshold"].items():
    print(f"  {k}: {float(v):.4f}")


In [ ]:
# Reconstruir la arquitectura y cargar los pesos guardados
model = timm.create_model(
    artifact["model_name"],
    pretrained=False,
    num_classes=artifact["num_classes_head"],
)
model.load_state_dict(artifact["state_dict"])
model.to(DEVICE)
model.eval()

print("Modelo cargado y listo para inferencia.")


## 2. Función de predicción

Aplica exactamente el mismo preprocesamiento (tamaño de imagen y normalización) que se usó
en el entrenamiento del modelo guardado -- estos valores vienen del propio artefacto, no hay
que adivinarlos ni volver a escribirlos.

In [ ]:
# Mismo preprocesamiento que en la evaluación: lado corto al tamaño de entrada y recorte central
# (no se estira la imagen a un cuadrado). Todos los valores vienen del propio artefacto.
transform = transforms.Compose([
    transforms.Resize(artifact["image_size"]),
    transforms.CenterCrop(artifact["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=artifact["normalize_mean"], std=artifact["normalize_std"]),
])
THRESHOLD = float(artifact["threshold"])  # umbral elegido en validación (no 0.5)

def predict_image(image: Image.Image, show=True):
    """Recibe una imagen PIL y devuelve (etiqueta, probabilidad de que sea generada por IA)."""
    if image.mode != "RGB":
        image = image.convert("RGB")

    x = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logit = model(x).squeeze(1)
        prob_generada = torch.sigmoid(logit).item()  # probabilidad de la clase 1 (generada)

    pred_class = 1 if prob_generada >= THRESHOLD else 0
    label = artifact["label_mapping"][pred_class]

    if show:
        plt.figure(figsize=(5, 5))
        plt.imshow(image)
        plt.axis("off")
        color = "seagreen" if pred_class == 0 else "crimson"
        plt.title(f"{label}  (prob. generada = {prob_generada:.4f}; umbral = {THRESHOLD:.4f})",
                  color=color, fontsize=11, fontweight="bold")
        plt.show()

    return label, prob_generada


## 3. Subir una imagen y clasificarla

Corre la celda de abajo: te va a pedir subir un archivo de imagen (jpg, png, etc.).
Funciona tanto en **Google Colab** como en entornos genéricos (Jupyter/Lightning AI Studio).

In [ ]:
def upload_image():
    """Sube una imagen desde tu computador. Compatible con Colab y con Jupyter/Lightning genérico."""
    try:
        # --- Camino Colab ---
        from google.colab import files
        uploaded = files.upload()
        filename = next(iter(uploaded))
        return Image.open(filename)
    except ImportError:
        # --- Camino genérico (Jupyter / Lightning AI Studio) vía ipywidgets ---
        import ipywidgets as widgets
        from IPython.display import display
        import io

        uploader = widgets.FileUpload(accept="image/*", multiple=False)
        display(uploader)
        print("Sube una imagen arriba y luego vuelve a correr esta celda para procesarla.")

        if len(uploader.value) == 0:
            return None

        file_info = list(uploader.value.values())[0]
        return Image.open(io.BytesIO(file_info["content"]))


img = upload_image()


In [ ]:
# Si usaste el uploader de ipywidgets (entorno genérico), corre esta celda DESPUÉS de subir el archivo arriba.
# Si estás en Colab, 'img' ya debería estar listo desde la celda anterior.

if img is not None:
    label, prob = predict_image(img)
    print(f"\nPredicción: {label}")
    print(f"Probabilidad de ser generada: {prob:.4f} (umbral {THRESHOLD:.4f})")
else:
    print("Aún no se ha cargado ninguna imagen. Sube una arriba y vuelve a correr esta celda.")


## 4. (Alternativo) Clasificar una imagen ya guardada en disco

Si prefieres no usar el uploader interactivo -- por ejemplo, para probar varias imágenes
de una carpeta -- puedes cargarlas directamente por ruta.

In [ ]:
# Ejemplo: clasificar una imagen por ruta local
IMAGE_PATH = "ruta/a/tu/imagen.jpg"  # <-- ajustar

if os.path.exists(IMAGE_PATH):
    img = Image.open(IMAGE_PATH)
    label, prob = predict_image(img)
    print(f"\nPredicción: {label} (prob. generada = {prob:.4f})")
else:
    print(f"Ajusta IMAGE_PATH -- no se encontró: {IMAGE_PATH}")


## 5. (Alternativo) Clasificar varias imágenes de una carpeta

Útil para hacer una prueba rápida sobre un lote de imágenes en vez de una por una.

In [ ]:
import glob

FOLDER_PATH = "ruta/a/carpeta/con/imagenes"  # <-- ajustar

image_paths = glob.glob(os.path.join(FOLDER_PATH, "*.jpg")) + \
              glob.glob(os.path.join(FOLDER_PATH, "*.png")) + \
              glob.glob(os.path.join(FOLDER_PATH, "*.jpeg"))

results = []
for path in image_paths:
    img = Image.open(path)
    label, prob = predict_image(img, show=False)
    results.append({"archivo": os.path.basename(path), "prediccion": label, "prob_generada": round(prob, 6)})

import pandas as pd
results_df = pd.DataFrame(results)
results_df


## Nota sobre confiabilidad

Este modelo fue entrenado sobre el dataset `Defactify_Image_Dataset` con generadores específicos.
Va a funcionar mejor en imágenes similares a las que vio en entrenamiento, y puede generalizar peor
a generadores muy recientes que no estaban en el dataset (ej. Nano Banana Pro, GPT Image 2, Midjourney V8).
Trata la predicción como una **señal orientativa**, no como una verificación forense definitiva.

Las etiquetas mostradas (`Real` / `Falso`) vienen del `label_mapping` guardado en el artefacto `.pt`
-- si en algún momento reentrenas con otro dataset, ese mapeo se actualiza solo, no hay que tocar este notebook.

**Sobre el umbral.** El umbral guardado (≈0.0024) se eligió con imágenes de validación sin alterar. Con imágenes
recomprimidas, con ruido o reducidas de tamaño, como las que circulan en redes sociales, el desempeño cambia
y conviene recalibrar el umbral con imágenes parecidas a las de uso (ver el análisis de robustez del trabajo).
